# 01 · Colab 上完成一次短实验，并把记录带回 Mac

本地电脑负责阅读、CPU 自检和结果分析；本脚本的真实训练在 **Colab CUDA GPU** 上运行。它是 SFT/DFT 链路检查，暂不代表选择了论文课题。

**默认不安装、不下载模型、不训练。** `RUN_INSTALL=False`、`RUN_TRAIN=False`。先使用 CPU runtime；准备实际训练时再切 GPU，结束后及时断开并删除运行时。已分配 GPU 时，仅运行 CPU 代码也可能消耗额度。

## 1. 上传专用包

在 Colab 左侧文件面板上传 **`PostTraining_Colab_v2.zip`**，下一格会解压为 `/content/PostTraining/`。不要只上传 ipynb，也不要上传整个 Mac 部署包。已有完整的 `PostTraining/` 则直接沿用，不覆盖文件。

示例为 `Qwen/Qwen3-0.6B-Base`，固定文本前缀、不使用聊天模板。依赖版本仍是**未通过实际 GPU 验收的候选组合**；保留 Colab 自带 PyTorch/CUDA，不在 Mac 或 `.venv-zotero` 安装这组训练依赖。

## 2. 时间与保存

每分支最多 20 步、batch=1、梯度累计=8、max length=512。完整 pair 的训练子进程默认 **20 分钟超时**，包括模型下载、加载、评估和训练，不包括依赖安装。超时或手动中断会停止进程组，并在 `finally` 中保存已有日志；失败记录不会变成实验结果。

每次运行使用唯一目录。每个分支完成后更新恢复 ZIP；成功或失败退出时再次归档。包内只含 LoRA adapter、tokenizer、配置、数据与源码快照、日志，不复制基础模型缓存。**不支持中途续训**，失败后换新 run 重跑。

`RECOVERY_DIR=None` 时 ZIP 仍在 Colab 临时盘，必须下载到 Mac。若你已挂载 Drive，可填入一个已存在的专用目录，恢复包会在分支完成和退出时复制过去；本 notebook 不替你登录或挂载账号。硬断连时 Python 的 `finally` 无法保证执行，最多恢复此前已复制的检查点。

默认 12/4 条手写加法样本只检查链路，不能证明 DFT 更好。BF16 GPU 使用 BF16，否则明确使用 FP32；未承诺具体 GPU 的显存、速度或额度消耗。


Local Mac validation on 2026-10-15: this copy was executed top-to-bottom in a real Jupyter kernel using the existing .venv-research environment. Dependency installation and training remained disabled. The two PyTorch checks were skipped because torch is absent; 15 other foundation checks passed. No model was downloaded, no GPU training ran, and no research result was produced. The original cloud notebook is preserved in the downloaded package.

In [1]:
from pathlib import Path, PurePosixPath
import sys
import zipfile
import stat

candidates = [Path.cwd(), Path.cwd().parent, Path('/content/PostTraining'), Path.cwd()/'PostTraining']
ROOT = next((p.resolve() for p in candidates if (p/'experiments/contracts.py').is_file()), None)
if ROOT is None:
    uploaded = next((p for p in [Path('/content/PostTraining_Colab_v2.zip'),
                                Path.cwd()/'PostTraining_Colab_v2.zip'] if p.is_file()), None)
    if uploaded is not None:
        destination = uploaded.parent
        target = destination/'PostTraining'
        if target.exists():
            raise FileExistsError('已有同名但不完整的目录；请检查或换新的 runtime，不覆盖原文件。')
        with zipfile.ZipFile(uploaded) as archive:
            for entry in archive.infolist():
                path = PurePosixPath(entry.filename)
                parts = path.parts
                if (path.is_absolute() or not parts or parts[0] != 'PostTraining'
                    or '..' in parts or chr(92) in entry.filename
                    or stat.S_ISLNK(entry.external_attr >> 16)):
                    raise ValueError('ZIP条目必须安全地位于PostTraining根目录')
            archive.extractall(destination)
        ROOT = target.resolve() if (target/'experiments/contracts.py').is_file() else None
if ROOT is None:
    raise FileNotFoundError('在左侧上传PostTraining_Colab_v2.zip后重跑本格。')
sys.path.insert(0, str(ROOT))
print('已找到实验包:', ROOT.name)

from experiments.recovery import unique_run_dir
RUN_INSTALL = False
RUN_TRAIN = False
TRAIN_JSONL = None
VAL_JSONL = None
MAX_STEPS = 20
MAX_LENGTH = 512
SEED = 42
PAIR_TIMEOUT_SECONDS = 20 * 60
# 例如 Path('/content/drive/MyDrive/Recovery')；必须先自行挂载并创建目录。
RECOVERY_DIR = None
RUN_DIR = unique_run_dir(ROOT/'runs')
print({'RUN_INSTALL':RUN_INSTALL, 'RUN_TRAIN':RUN_TRAIN,
       'max_steps_per_branch':MAX_STEPS, 'pair_timeout_seconds':PAIR_TIMEOUT_SECONDS,
       'dataset':'synthetic_smoke_only' if TRAIN_JSONL is None else 'local_jsonl'})

import json
from experiments.preflight import environment_report
environment = environment_report()
print(json.dumps(environment, ensure_ascii=False, indent=2))


已找到实验包: PostTraining
{'RUN_INSTALL': False, 'RUN_TRAIN': False, 'max_steps_per_branch': 20, 'pair_timeout_seconds': 1200, 'dataset': 'synthetic_smoke_only'}
{
  "recorded_utc": "2026-10-25T20:28:46.773864+00:00",
  "python": "3.12.13",
  "platform": "macOS-27.0.1-arm64-arm-64bit",
  "machine": "arm64",
  "working_disk_free_bytes": 112105803776,
  "mps_built": false,
  "mps_available": false,
  "packages": {
    "numpy": "2.5.4",
    "torch": null,
    "transformers": null,
    "peft": null,
    "accelerate": null,
    "huggingface-hub": null,
    "tokenizers": null,
    "safetensors": null
  },
  "cuda_available": false,
  "gpu": null
}


## 3. 可选安装，然后通过 CPU 自检

安装只在本次独立 Colab runtime 中进行。若更改了已加载的包，请重启 runtime，关闭 `RUN_INSTALL`，从第一格重跑。安装不属于 20 分钟训练超时范围。

自检不下载模型。原有检查核对 mask、shift、梯度和数据边界；恢复测试只启动短小的 Python 子进程，检查失败与超时日志能否归档，不运行训练。没有 torch 时真实 autograd 检查明确显示 skipped。


In [2]:
import subprocess
if RUN_INSTALL:
    subprocess.run([sys.executable,'-m','pip','install','-r',str(ROOT/'experiments/requirements-colab.txt')], check=True)
else:
    print('跳过安装。')

import unittest
suite = unittest.defaultTestLoader.discover(str(ROOT/'tests'), pattern='test_*.py')
checks = unittest.TextTestRunner(verbosity=1).run(suite)
assert checks.wasSuccessful()
print({'tests_run':checks.testsRun,'skipped':len(checks.skipped)})

.

.

.

.

.

...

.

跳过安装。


.

----------------------------------------------------------------------
Ran 17 tests in 0.258s

OK (skipped=2)


{'tests_run': 17, 'skipped': 2}


## 4. 运行或只看预检

两分支共享解析到完整 SHA 的基础模型、tokenizer、数据、初始化和样本顺序。源码快照能随结果包回到 Mac，控制台输出保存为 `console.log`；`launcher.json` 记录超时、退出状态和总墙钟时间。

比较用同一验证样本和 greedy 解码。不同优化损失不能直接比数值；只有一个 seed、极小合成数据和未调优学习率，这不是算法优劣实验。

开启训练后，当前单元格会实时显示子进程日志。失败或超时会显示异常，这是保留失败证据的正常方式；恢复 ZIP 会在异常抛出前生成，不需要后续单元格成功运行。


In [3]:
command = [sys.executable,'-u','-m','experiments.run_pair','--output',str(RUN_DIR),
           '--max-steps',str(MAX_STEPS),'--max-length',str(MAX_LENGTH),'--seed',str(SEED)]
if TRAIN_JSONL is not None or VAL_JSONL is not None:
    assert TRAIN_JSONL and VAL_JSONL, '需要同时提供训练和验证JSONL'
    command += ['--train-jsonl',str(TRAIN_JSONL),'--val-jsonl',str(VAL_JSONL)]
if RECOVERY_DIR is not None:
    command += ['--recovery-dir',str(RECOVERY_DIR)]
if RUN_TRAIN:
    assert checks.wasSuccessful()
    command += ['--confirm-smoke-training']
    from experiments.recovery import run_logged_process
    print('开始受限联调；超时、失败和中断均尝试保留已有记录。')
    launch = run_logged_process(command, cwd=ROOT, run_dir=RUN_DIR,
                                timeout_seconds=PAIR_TIMEOUT_SECONDS, recovery_dir=RECOVERY_DIR)
else:
    completed = subprocess.run(command, cwd=ROOT, check=True, text=True, capture_output=True)
    preflight = json.loads(completed.stdout)
    print({k:preflight[k] for k in ('status','train_examples','validation_examples')})


{'status': 'preflight_only_no_download_no_training', 'train_examples': 12, 'validation_examples': 4}


## 5. 查看结果和恢复包

只有两分支完成且 manifest 仍标记成功，才展示对照；失败和中断只展示运行状态。`training_seconds` 仅覆盖优化更新；`branch_wall_seconds` 包含加载与评估，`launcher.json` 的总耗时覆盖完整 pair。它们都不是 Colab 额度账单。

在左侧文件面板找到 `runs/<本次run名>.zip` 并下载到 Mac。它包含原始数据快照，按数据本身的分享权限处理。**生成归档不等于已经下载。** 如上一格失败，ZIP 也已在 `finally` 中尝试生成，可以直接下载，无需重新训练。


In [4]:
comparison = None
manifest_path = RUN_DIR/'manifest.json'
manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else None
if (RUN_TRAIN and manifest and manifest.get('status') == 'smoke_complete_not_research_evidence'
    and (RUN_DIR/'comparison.json').is_file()):
    comparison = json.loads((RUN_DIR/'comparison.json').read_text())
    for row in comparison['results']:
        print({'objective':row['objective'],
               'before_validation_nll':row['before_validation']['unweighted_nll'],
               'after_validation_nll':row['after_validation']['unweighted_nll'],
               'training_seconds':row['training_seconds'], 'branch_wall_seconds':row['branch_wall_seconds'],
               'peak_cuda_allocated_GiB':row['peak_cuda_allocated_bytes']/2**30})
else:
    print('运行状态:', manifest.get('status') if manifest else '未运行真实训练')
    print('没有完整对照，不报告模型效果。')
archive = RUN_DIR.parent / f'{RUN_DIR.name}.zip'
if archive.is_file():
    print('可下载恢复包:', archive)
    if RECOVERY_DIR is None:
        print('仍在当前运行时；请下载到Mac。')
else:
    print('没有运行恢复包；默认预检不会创建模型实验记录。')


运行状态: 未运行真实训练
没有完整对照，不报告模型效果。
没有运行恢复包；默认预检不会创建模型实验记录。


## 下一步

回到 Mac，ZIP 可以保存在 Downloads 或 `workbench-v2/inbox/`。在 `the extracted repository root` 运行 `python3 tools/research_files.py import-run /实际路径/本次run.zip`；导入到 `workbench-v2/runs/`，不覆盖既有结果。阅读 `launcher.json`、manifest、样本剔除和生成前后输出；未成功的 pair 保留为失败记录，不参与方法比较。

官方资料：[Qwen 模型卡](https://huggingface.co/Qwen/Qwen3-0.6B-Base) · [Transformers 4.51.3](https://huggingface.co/docs/transformers/v4.51.3/en/main_classes/model) · [PEFT LoRA](https://huggingface.co/docs/peft/v0.15.0/en/package_reference/lora) · [TRL DFT](https://huggingface.co/docs/trl/en/sft_trainer)
